In [ ]:
# Cell 104 (02_structure, cell 1): CONFIG + load Phase 2 frames + input asserts (READS ONLY; writes nothing)
import os, sys, json, pickle, hashlib, importlib
import numpy as np, pandas as pd

assert os.environ.get('PYTHONHASHSEED') == '0', "PYTHONHASHSEED is not '0': wrong kernel, STOP"
SEED = 42
PREFIX = 'sel_'
ROOT = os.environ["CREDIT_BN_ROOT"]
PROC = ROOT + "/Data/processed/"
sys.path.insert(0, ROOT + "/shared/src")
L = lambda p: pickle.load(open(p, 'rb'))

node_order = json.load(open(PROC + "sel_node_order.json"))
state_names = json.load(open(PROC + "sel_state_names.json"))
NODES7 = node_order['derived_7_nodes_selection']
NODES9 = node_order['all_9_nodes']
IND_EDGES = [tuple(e) for e in node_order['indicator_edges']]
print("7 nodes:", NODES7)
print("9 nodes:", NODES9)
print("indicator edges:", IND_EDGES)

sel_train_cc = L(PROC + "sel_train_cc.pkl")
assert sel_train_cc.shape == (96251, 9)
data7 = sel_train_cc[NODES7].copy()
assert list(data7.columns) == NODES7
assert data7.notna().all().all()
print("\ndata7:", data7.shape, "| NaN:", int(data7.isna().sum().sum()))
print("indicators constant on complete cases:", {c: sorted(sel_train_cc[c].unique().tolist()) for c in ['M_Income', 'M_Dependents']})
print("dtypes:", {c: str(data7[c].dtype) for c in data7.columns})
print("states per node:", {c: int(data7[c].nunique()) for c in data7.columns})
print("Default rate:", round(float(data7['Default'].mean()), 6))

sn = importlib.import_module("structure")
print("\nstructure.learn_structure available:", hasattr(sn, "learn_structure"))

In [ ]:
# Cell 105 (02_structure, cell 2): STRUCTURE LEARNING on the 7 selection nodes (READS ONLY; writes nothing; results kept in kernel variables)
import warnings, time
warnings.filterwarnings("ignore")
from pgmpy.estimators import BIC
from pgmpy.base import DAG

importlib.reload(sn)
res7 = sn.learn_structure(data7, NODES7, seeds=range(42, 57), edge_density=0.15, verbose=False)

def ehash(edges):
    return hashlib.sha256(json.dumps(sorted(map(list, edges))).encode()).hexdigest()[:8]

rows = []
for score, dag, seed in res7['results']:
    ed = sorted(map(tuple, dag.edges()))
    rows.append(dict(seed=seed, BIC=float(score), n_edges=len(ed), edge_hash=ehash(ed)))
seed_tbl = pd.DataFrame(rows)
best = seed_tbl['BIC'].max()
seed_tbl['is_best'] = np.isclose(seed_tbl['BIC'], best, rtol=0, atol=1e-6)
pd.options.display.float_format = '{:.4f}'.format
print(seed_tbl.to_string(index=False))

best_edges = res7['best_edges']
best_hash = ehash(best_edges)
n_best = int(seed_tbl['is_best'].sum())
n_distinct = seed_tbl['edge_hash'].nunique()
other = seed_tbl[seed_tbl['edge_hash'] != best_hash]
gap = (best - other['BIC'].max()) if len(other) else None
bic = BIC(data7)
empty = DAG(); empty.add_nodes_from(NODES7)
print(f"\nbest BIC: {best:.4f} (seed {res7['best_seed']}, hash {best_hash}) | seeds reaching best BIC: {n_best}/15")
print(f"distinct structures (by edge-set hash): {n_distinct}")
print("BIC gap to best structure that differs from the winner:", None if gap is None else round(float(gap), 4))
print(f"BIC of empty graph: {bic.score(empty):.4f}")
print(f"runtime: {res7['runtime_s']:.0f}s")
print(f"\nbest structure: {len(best_edges)} edges")
for e in best_edges:
    print("  ", e[0], "->", e[1])

In [ ]:
# Cell 106 (02_structure, cell 3): 9-node DAG, Markov blankets, comparison with locked, SAVE (sel_). Needs Cell 105 variables.
# WRITES (new, safe_save): selection/tables/sel_structure_7node_edges.json, sel_dag_9node_edges.json, sel_structure_seed_table.csv,
#                          sel_markov_blanket_report.json, selection/figures/sel_dag_9node.png
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from safe_save import safe_save

edges7 = [tuple(e) for e in best_edges]
edges9 = edges7 + IND_EDGES
G7, G9 = nx.DiGraph(), nx.DiGraph()
G7.add_nodes_from(NODES7); G7.add_edges_from(edges7)
G9.add_nodes_from(NODES9); G9.add_edges_from(edges9)
assert nx.is_directed_acyclic_graph(G7) and nx.is_directed_acyclic_graph(G9)
assert len(edges9) == len(edges7) + 2
assert set(G9.nodes()) == set(NODES9) and set(G7.nodes()) == set(NODES7)
assert G9.in_degree('M_Income') == 0 and G9.in_degree('M_Dependents') == 0, "indicators must be roots"
assert G7.degree('Default') > 0, "Default isolated: STOP"
print(f"7-node: {len(edges7)} edges | 9-node: {len(edges9)} edges | acyclic: True | 9 nodes: True | indicators are roots: True")

def blanket(G, t='Default'):
    pa = sorted(G.predecessors(t)); ch = sorted(G.successors(t))
    co = sorted({p for c in ch for p in G.predecessors(c)} - {t})
    return dict(parents=pa, children=ch, co_parents=co, blanket=sorted(set(pa) | set(ch) | set(co)))
B7, B9 = blanket(G7), blanket(G9)
KEEP = [n for n in NODES7 if n != 'Default']
for nm, B in [('7-node', B7), ('9-node', B9)]:
    print(f"\n{nm} Default: parents={B['parents']} children={B['children']} co-parents={B['co_parents']}")
    print(f"   blanket = {B['blanket']}")
inside = [v for v in KEEP if v in B9['blanket']]
outside = [v for v in NODES9 if v != 'Default' and v not in B9['blanket']]
print("\nkept variables INSIDE blanket:", inside)
print("M_Income in blanket:", 'M_Income' in B9['blanket'], "| M_Dependents in blanket:", 'M_Dependents' in B9['blanket'])
print("OUTSIDE blanket (exactly zero sensitivity):", outside)
can_miss = [v for v in B9['blanket'] if v in ('Monthly Income (bin)', 'Number of Dependents (bin)')]
print("blanket variables that can be missing at inference:", can_miss, "| always observed:", [v for v in B9['blanket'] if v not in can_miss])

locked = L(ROOT + "/full_variable/models/bn_final_model_iteration79.pkl")
IND = set(IND_EDGES)
lk = {tuple(e) for e in locked['model_edges']} - IND
lk7 = {e for e in lk if e[0] in NODES7 and e[1] in NODES7}
s7 = set(edges7)
cmp_ = dict(shared_same_direction=sorted(s7 & lk7), new_in_selection=sorted(s7 - lk7), locked_absent_in_selection=sorted(lk7 - s7),
            reversed=sorted(e for e in s7 - lk7 if (e[1], e[0]) in lk7))
print(f"\nlocked 22 edges; {len(lk7)} of them are among the 7 shared nodes")
for k, v in cmp_.items():
    print(f"  {k} ({len(v)}):"); [print("     ", a, "->", b) for a, b in v]

report = dict(blanket_7node=B7, blanket_9node=B9, kept_inside_blanket=inside, indicators_in_blanket=[i for i in ('M_Income', 'M_Dependents') if i in B9['blanket']],
              outside_blanket_zero_sensitivity=outside, blanket_vars_can_be_missing=can_miss,
              blanket_vars_always_observed=[v for v in B9['blanket'] if v not in can_miss], comparison_with_locked=cmp_,
              best_seed=int(res7['best_seed']), best_bic=float(best), seeds_reaching_best=n_best, distinct_structures=int(n_distinct),
              bic_gap_to_best_different=float(gap), bic_empty_graph=float(bic.score(empty)))

pos = {'Default': (0, 0), 'Age (bin)': (-2, 1.3), 'Debt Ratio (bin)': (2, 1.3), 'Monthly Income (bin)': (4.2, 3.2), 'Open Credit Lines (bin)': (0, 3.2),
       'Real Estate Loans (bin)': (-4.2, 3.2), 'Number of Dependents (bin)': (-4.2, 5.2), 'M_Income': (4.2, 5.2), 'M_Dependents': (-6.4, 5.2)}
fig, ax = plt.subplots(figsize=(12, 7.5))
col = {n: ('#e45756' if n == 'Default' else '#f2b134' if n in B9['blanket'] else '#d9d9d9') for n in NODES9}
nx.draw_networkx_nodes(G9, pos, nodelist=[n for n in NODES9 if not n.startswith('M_')], node_color=[col[n] for n in NODES9 if not n.startswith('M_')],
                       node_size=3600, edgecolors='black', ax=ax)
nx.draw_networkx_nodes(G9, pos, nodelist=['M_Income', 'M_Dependents'], node_color='#bcd4f2', node_shape='s', node_size=3000, edgecolors='#1f4e9c', ax=ax)
nx.draw_networkx_edges(G9, pos, edgelist=edges7, arrows=True, arrowsize=18, node_size=3600, width=1.6, ax=ax)
nx.draw_networkx_edges(G9, pos, edgelist=IND_EDGES, arrows=True, arrowsize=18, node_size=3000, style='dashed', edge_color='#1f4e9c', width=1.6, ax=ax)
nx.draw_networkx_labels(G9, pos, labels={n: n.replace(' (bin)', '').replace(' ', '\n', 1) for n in NODES9}, font_size=8, ax=ax)
ax.set_title("Selection BN (9 nodes). Red = Default; orange = Default's Markov blanket; grey = outside blanket; blue squares = missingness indicators")
ax.axis('off'); ax.set_xlim(-8, 6)

for obj, rel, kw in [(dict(edges=[list(e) for e in edges7], bic=float(best), seed=int(res7['best_seed'])), "selection/tables/sel_structure_7node_edges.json", {}),
                     (dict(edges=[list(e) for e in edges9], nodes=NODES9), "selection/tables/sel_dag_9node_edges.json", {}),
                     (seed_tbl, "selection/tables/sel_structure_seed_table.csv", {"index": False}),
                     (report, "selection/tables/sel_markov_blanket_report.json", {}),
                     (fig, "selection/figures/sel_dag_9node.png", {})]:
    path, h = safe_save(obj, ROOT + "/" + rel, **kw)
    print("saved:", rel, h[:12])
plt.show()

In [ ]:
# Cell 107 (02_structure, cell 4): READ-ONLY -- find the old cell(s) that drew the full-model DAG
import re
hist = open(ROOT + "/archive/kernel/In_history_at_cell78.txt").read()
parts = re.split(r"# ===== In\[(\d+)\] =====\n", hist)
blocks = {int(parts[i]): parts[i + 1] for i in range(1, len(parts) - 1, 2)}

def active(c):
    return "\n".join(l for l in c.splitlines() if l.strip() and not l.lstrip().startswith('#'))

hits = [i for i, c in sorted(blocks.items()) if i < 74
        and 'savefig' in active(c) and re.search(r"dag|DAG|node_color|draw_networkx|FancyArrow|annotate", active(c))]
print("history cells that draw and save a DAG-style figure:", hits)
for i in hits:
    lines = blocks[i].splitlines()
    print(f"\n===== In[{i}] ({len(lines)} lines; showing up to 120) =====")
    print("\n".join(lines[:120]))

In [ ]:
# Cell 108 (02_structure, cell 6): REDRAW 9-node DAG in the full-model style (WRITES one new file via safe_save: selection/figures/sel_dag_9node_v2.png)
import networkx as nx
import matplotlib.pyplot as plt
from safe_save import safe_save
get_ipython().run_line_magic('matplotlib', 'inline')

bn_model = G9
learned_edges = edges7
fixed_edges = IND_EDGES
indicator_nodes = {'M_Income', 'M_Dependents'}
target_node = {'Default'}
substantive_nodes = [n for n in bn_model.nodes() if n not in indicator_nodes and n not in target_node]
blanket_nodes = [n for n in B9['blanket']]
other_nodes = [n for n in substantive_nodes if n not in blanket_nodes]

fig, ax = plt.subplots(figsize=(20, 12))
try:
    pos = nx.nx_agraph.graphviz_layout(bn_model, prog='dot', args='-Gnodesep=1.0 -Granksep=1.0')
    print("layout: graphviz dot")
except Exception as ex:
    print("graphviz layout unavailable (", type(ex).__name__, ") -> using fallback layout")
    pos = {'Default': (0, 0), 'Age (bin)': (-2, 1.3), 'Debt Ratio (bin)': (2, 1.3), 'Monthly Income (bin)': (4.2, 3.2),
           'Open Credit Lines (bin)': (0, 3.2), 'Real Estate Loans (bin)': (-4.2, 3.2), 'Number of Dependents (bin)': (-4.2, 5.2),
           'M_Income': (4.2, 5.2), 'M_Dependents': (-6.4, 5.2)}
    pos = {k: (v[0] * 200, v[1] * 200) for k, v in pos.items()}

node_size = 5000
nx.draw_networkx_nodes(bn_model, pos, nodelist=other_nodes, node_color='lightblue', node_size=node_size,
                       edgecolors='black', ax=ax)
nx.draw_networkx_nodes(bn_model, pos, nodelist=blanket_nodes, node_color='lightblue', node_size=node_size,
                       edgecolors='darkorange', linewidths=4, ax=ax)
nx.draw_networkx_nodes(bn_model, pos, nodelist=list(target_node), node_color='gold', node_size=node_size,
                       edgecolors='black', linewidths=2, ax=ax)
nx.draw_networkx_nodes(bn_model, pos, nodelist=list(indicator_nodes), node_color='lightgray', node_size=node_size,
                       edgecolors='black', node_shape='s', ax=ax)

nx.draw_networkx_edges(bn_model, pos, edgelist=learned_edges, edge_color='gray', arrows=True, arrowstyle='-|>',
                       arrowsize=18, node_size=node_size, connectionstyle='arc3,rad=0.08', ax=ax)
nx.draw_networkx_edges(bn_model, pos, edgelist=fixed_edges, edge_color='dimgray', arrows=True, arrowstyle='-|>',
                       arrowsize=18, node_size=node_size, width=2.2, style='dashed', ax=ax)

labels = {n: n.replace(' (bin)', '') for n in bn_model.nodes()}
nx.draw_networkx_labels(bn_model, pos, labels=labels, font_size=8, font_weight='bold', ax=ax)

ax.set_title('9-Node Selection Bayesian Network Structure (application-stage variables)\n'
             '(Yellow = target (Default), Blue = substantive features, Orange outline = Default\'s Markov blanket,\n'
             'Gray squares = missingness indicators (root nodes))', fontsize=13)
ax.axis('off')
plt.tight_layout()

path, h = safe_save(fig, ROOT + "/selection/figures/sel_dag_9node_v2.png", dpi=300)
print("saved: selection/figures/sel_dag_9node_v2.png", h[:12])
plt.show()

In [ ]:
# Cell 109 (02_structure, cell 7): REDRAW v3 (readable labels). WRITES one new file via safe_save: selection/figures/sel_dag_9node_v3.png
import textwrap
import networkx as nx
import matplotlib.pyplot as plt
from safe_save import safe_save
get_ipython().run_line_magic('matplotlib', 'inline')

bn_model = G9
learned_edges = edges7
fixed_edges = IND_EDGES
indicator_nodes = {'M_Income', 'M_Dependents'}
target_node = {'Default'}
substantive_nodes = [n for n in bn_model.nodes() if n not in indicator_nodes and n not in target_node]
blanket_nodes = list(B9['blanket'])
other_nodes = [n for n in substantive_nodes if n not in blanket_nodes]

fig, ax = plt.subplots(figsize=(22, 15))
try:
    pos = nx.nx_agraph.graphviz_layout(bn_model, prog='dot', args='-Gnodesep=1.6 -Granksep=1.6')
    print("layout: graphviz dot")
except Exception as ex:
    print("graphviz layout unavailable (", type(ex).__name__, ") -> fallback layout")
    pos = {'Default': (0, 0), 'Age (bin)': (-2, 1.3), 'Debt Ratio (bin)': (2, 1.3), 'Monthly Income (bin)': (4.2, 3.2),
           'Open Credit Lines (bin)': (0, 3.2), 'Real Estate Loans (bin)': (-4.2, 3.2), 'Number of Dependents (bin)': (-4.2, 5.2),
           'M_Income': (4.2, 5.2), 'M_Dependents': (-6.4, 5.2)}
    pos = {k: (v[0] * 250, v[1] * 250) for k, v in pos.items()}

node_size = 9000
nx.draw_networkx_nodes(bn_model, pos, nodelist=other_nodes, node_color='lightblue', node_size=node_size,
                       edgecolors='black', ax=ax)
nx.draw_networkx_nodes(bn_model, pos, nodelist=blanket_nodes, node_color='lightblue', node_size=node_size,
                       edgecolors='darkorange', linewidths=5, ax=ax)
nx.draw_networkx_nodes(bn_model, pos, nodelist=list(target_node), node_color='gold', node_size=node_size,
                       edgecolors='black', linewidths=2, ax=ax)
nx.draw_networkx_nodes(bn_model, pos, nodelist=list(indicator_nodes), node_color='lightgray', node_size=node_size,
                       edgecolors='black', node_shape='s', ax=ax)

nx.draw_networkx_edges(bn_model, pos, edgelist=learned_edges, edge_color='gray', arrows=True, arrowstyle='-|>',
                       arrowsize=20, node_size=node_size, connectionstyle='arc3,rad=0.08', ax=ax)
nx.draw_networkx_edges(bn_model, pos, edgelist=fixed_edges, edge_color='dimgray', arrows=True, arrowstyle='-|>',
                       arrowsize=20, node_size=node_size, width=2.2, style='dashed', ax=ax)

def lab(n):
    s = n.replace(' (bin)', '').replace('_', '_ ') if n.startswith('M_') else n.replace(' (bin)', '')
    return textwrap.fill(s, width=11) if not n.startswith('M_') else s.replace('_ ', '_\n')
labels = {n: lab(n) for n in bn_model.nodes()}
nx.draw_networkx_labels(bn_model, pos, labels=labels, font_size=9, font_weight='bold', ax=ax)

ax.set_title('9-Node Selection Bayesian Network Structure (application-stage variables)\n'
             '(Yellow = target (Default), Blue = substantive features, Orange outline = Default\'s Markov blanket,\n'
             'Gray squares = missingness indicators (root nodes))', fontsize=14)
ax.axis('off')
ax.margins(0.08)
plt.tight_layout()

path, h = safe_save(fig, ROOT + "/selection/figures/sel_dag_9node_v3.png", dpi=300)
print("saved: selection/figures/sel_dag_9node_v3.png", h[:12])
plt.show()